# Dynamic flight model: vertical velocities

Run cells from top to bottom. Speeds use **cm/s**, positive upward. The model returns m/s and is converted to cm/s before comparison. Select a local dataset in the configuration cell; the example path is retained from the original notebook. The basestation `GLIDER_VERT_VELO_MODEL` is assumed to be in cm/s (adjust `BASESTATION_TO_CM_S` if its metadata says otherwise).

## 1. Load and prepare the dataset

In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, interactive_output
from IPython.display import display
from gliderflightOG1 import new_flightmodel as nfm, tools, utilities

DATASET_PATH = Path('/Users/u302042/Desktop/Master_thesis/Data/Glider/005/20080606/all_data_OG1.nc')
BASESTATION_TO_CM_S = 1.0  # set to 100 if GLIDER_VERT_VELO_MODEL is stored in m/s

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Set DATASET_PATH to an existing NetCDF file: {DATASET_PATH}')
glider = xr.open_dataset(DATASET_PATH).copy()
if 'VBD' not in glider:
    for candidate in ('VBD_CC', 'RELVBD'):
        if candidate in glider:
            glider = glider.rename({candidate: 'VBD'})
            break
if 'LATITUDE' not in glider and {'LAT', 'LON'} <= set(glider.variables):
    glider = glider.rename({'LAT': 'LATITUDE', 'LON': 'LONGITUDE'})
if 'C_VBD' not in glider:
    glider['C_VBD'] = xr.full_like(glider['VBD'], 1000.0)

# The depth derivative is negative on ascent; upward velocity is positive.
valid_depth = np.isfinite(glider['DEPTH'].values)
w_meas = np.full(glider.sizes['N_MEASUREMENTS'], np.nan)
w_meas[valid_depth] = -100 * tools.time_derivative(
    glider['TIME'].values[valid_depth], glider['DEPTH'].values[valid_depth], n=2
)
glider['W_MEAS'] = ('N_MEASUREMENTS', w_meas, {'units': 'cm/s', 'long_name': 'Measured vertical glider velocity'})
for name in ('VBD', 'PITCH', 'ROLL'):
    glider = utilities.interpolate_over_nans(glider, name)
glider['UPDN'] = xr.where(glider['PROFILE_NUMBER'] % 2 == 1, -1, 1)
print(f"Loaded {glider.sizes['N_MEASUREMENTS']:,} measurements")

## 2. Fit the flight model and calculate velocities

In [ ]:
def scalar(name):
    return float(glider[name].mean(skipna=True).values)

params = nfm.FlightParameters()
params.hd_a = scalar('HD_A')
params.hd_b = scalar('HD_B')
params.hd_c = scalar('HD_C')
params.vbd_min_cnts = scalar('VBD_MIN_CNTS')
params.vbd_cnts_per_cc = 1 / scalar('VBD_CC_PER_CNTS')
params.volmax = scalar('VOLMAX')
params.mass = scalar('MASS')
params.temp_ref = 10.0
params.rho0 = 1027.5
fm = nfm.SteadyFlightModel(params=params)

# Fit on finite input rows, retaining the same mask for restored outputs.
required = ('PRES', 'PSAL', 'TEMP', 'PITCH', 'VBD', 'C_VBD',
            'LATITUDE', 'LONGITUDE', 'W_MEAS', 'DEPTH')
valid_mask = np.logical_and.reduce([np.isfinite(glider[name].values) for name in required])
if not valid_mask.any():
    raise ValueError('No complete measurements are available for the flight model.')
fit_data = glider.isel(N_MEASUREMENTS=np.flatnonzero(valid_mask)).copy()
# The optimizer uses m/s internally; keep the published dataset in cm/s.
fit_data['W_MEAS'] = fit_data.W_MEAS / 100
fm.solve_model(fit_data, which_par=np.array([1, 1, 1, 0, 0, 0]))
speed_ms, angle_deg = fm.compute_flight_speed(fit_data)
speed_cm_s = np.full(glider.sizes['N_MEASUREMENTS'], np.nan)
angle_full_deg = np.full(glider.sizes['N_MEASUREMENTS'], np.nan)
speed_cm_s[valid_mask] = 100 * np.asarray(speed_ms)
angle_full_deg[valid_mask] = np.asarray(angle_deg)
model_cm_s = np.full(glider.sizes['N_MEASUREMENTS'], np.nan)
model_cm_s[valid_mask] = 100 * np.asarray(speed_ms) * np.sin(np.deg2rad(angle_deg))
glider['W_MODEL'] = ('N_MEASUREMENTS', model_cm_s, {'units': 'cm/s', 'long_name': 'Flight model vertical glider velocity'})

if 'GLIDER_VERT_VELO_MODEL' not in glider:
    raise KeyError('The dataset needs GLIDER_VERT_VELO_MODEL for the basestation comparison.')
bs_cm_s = glider['GLIDER_VERT_VELO_MODEL'] * BASESTATION_TO_CM_S
glider['W_BASESTATION'] = bs_cm_s.assign_attrs(units='cm/s', long_name='Basestation vertical glider velocity')
glider['W_WATER'] = (glider.W_MEAS - glider.W_MODEL).assign_attrs(units='cm/s', long_name='Water velocity calculated with flight model')
glider['W_WATER_BS'] = (glider.W_MEAS - glider.W_BASESTATION).assign_attrs(units='cm/s', long_name='Water velocity calculated with basestation model')
print(f'Model fitted on {valid_mask.sum():,} measurements')

## 3. Interactive flight model time series

Use **Start index** to choose the first measurement and **Points** to set the window length. The window stops at the end of the dataset. Blue always denotes this flight model, orange the basestation, and dark gray the measured velocity. Solid and dashed lines distinguish the two model estimates. All speeds are in cm/s (positive upward for vertical velocity). For zoom and pan in Jupyter, enable `%matplotlib widget` if `ipympl` is installed.

In [ ]:
N = glider.sizes['N_MEASUREMENTS']
if N < 1:
    raise ValueError('No measurements available to plot.')
start_widget = IntSlider(value=min(50000, N - 1), min=0, max=N - 1,
                         step=1, description='Start index:',
                         continuous_update=False, layout={'width': '520px'})
count_widget = IntSlider(value=min(4000, N), min=1, max=min(20000, N),
                         step=1, description='Points:',
                         continuous_update=False, layout={'width': '520px'})

MODEL_COLOR = '#0072B2'       # blue
BASESTATION_COLOR = '#D55E00' # vermilion
MEASURED_COLOR = '#333333'    # dark gray
ANGLE_COLOR = '#009E73'       # green, distinct from both models

def plot_window(start, points):
    sel = slice(start, min(start + points, N))
    time = glider.TIME.values[sel]
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True,
                             constrained_layout=True)

    # Glide speed and angle have different units, so they use separate y axes.
    ax = axes[0]
    speed_line, = ax.plot(time, speed_cm_s[sel], color=MODEL_COLOR,
                          lw=1.2, label='Flight model glide speed')
    ax.set_ylabel('Glide speed (cm/s)', color=MODEL_COLOR)
    ax.tick_params(axis='y', labelcolor=MODEL_COLOR)
    angle_axis = ax.twinx()
    angle_line, = angle_axis.plot(time, angle_full_deg[sel], color=ANGLE_COLOR,
                                  ls='--', lw=1.2, label='Flight model glide angle')
    angle_axis.set_ylabel('Glide angle (°)', color=ANGLE_COLOR)
    angle_axis.tick_params(axis='y', labelcolor=ANGLE_COLOR)
    ax.legend([speed_line, angle_line], [speed_line.get_label(), angle_line.get_label()],
              loc='upper right', fontsize=9)
    ax.set_title('Flight model speed and angle')

    ax = axes[1]
    ax.plot(time, glider.W_MEAS.values[sel], color=MEASURED_COLOR, lw=1.1,
            alpha=.8, label='Measured glider')
    ax.plot(time, glider.W_MODEL.values[sel], color=MODEL_COLOR, lw=1.3,
            label='Flight model')
    ax.plot(time, glider.W_BASESTATION.values[sel], color=BASESTATION_COLOR,
            ls='--', lw=1.3, label='Basestation model')
    ax.set_title('Glider vertical velocity')
    ax.set_ylabel('Vertical velocity (cm/s)')
    ax.legend(loc='upper right', fontsize=9)

    ax = axes[2]
    ax.plot(time, glider.W_WATER.values[sel], color=MODEL_COLOR, lw=1.3,
            label='Water: measured − flight model')
    ax.plot(time, glider.W_WATER_BS.values[sel], color=BASESTATION_COLOR,
            ls='--', lw=1.3, label='Water: measured − basestation')
    ax.set_title('Calculated vertical water velocity')
    ax.set_ylabel('Water velocity (cm/s)')
    ax.set_xlabel('Time')
    ax.legend(loc='upper right', fontsize=9)

    for ax in axes:
        ax.axhline(0, color='0.6', lw=.8, zorder=0)
        ax.grid(alpha=.2)
    plt.show()

display(VBox([start_widget, count_widget, interactive_output(
    plot_window, {'start': start_widget, 'points': count_widget})]))

## 4. Distributions of vertical velocity

The left panel compares measured glider velocity with the flight model and basestation estimates. The right panel compares the two calculated water velocities. Dashed lines show medians; the legend gives sample counts. Each histogram uses finite values and density scaling to allow comparisons when coverage differs.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
groups = [
    [('W_MEAS', 'Measured glider', 'black'), ('W_MODEL', 'Flight model', '#0072B2'),
     ('W_BASESTATION', 'Basestation model', '#D55E00')],
    [('W_WATER', 'Water: flight model residual', '#0072B2'),
     ('W_WATER_BS', 'Water: basestation residual', '#D55E00')],
]
for ax, group, title in zip(axes, groups, ('Glider velocity', 'Calculated water velocity')):
    values = [glider[name].values[np.isfinite(glider[name].values)] for name, _, _ in group]
    pooled = np.concatenate([v for v in values if len(v)])
    bins = np.histogram_bin_edges(pooled, bins=50)
    for (name, label, color), v in zip(group, values):
        if not len(v):
            continue
        ax.hist(v, bins=bins, density=True, histtype='step', linewidth=1.7,
                color=color, label=f'{label} (n={len(v):,})')
        ax.axvline(np.median(v), color=color, linestyle='--', alpha=.8)
    ax.axvline(0, color='0.6', linewidth=.8)
    ax.set(title=title, xlabel='Vertical velocity (cm/s)', ylabel='Density')
    ax.grid(alpha=.2)
    ax.legend(fontsize=8)
plt.show()

## 5. Split vertical profile near the surface and apogee

The split panels show water velocity calculated from this flight model, separately for descent and ascent. Surface depth and distance above each profile’s deepest point are binned independently. Curves show the mean and median, shading shows ±1 standard deviation, and dotted bounds show the 2.5th and 97.5th percentiles. Only bins with at least five finite measurements are drawn.

In [ ]:
depth = glider.DEPTH.values
profile = glider.PROFILE_NUMBER.values
height_above_apogee = np.full(depth.shape, np.nan, dtype=float)
for p in np.unique(profile[np.isfinite(profile)]):
    mask = (profile == p) & np.isfinite(depth)
    if mask.any():
        height_above_apogee[mask] = np.max(depth[mask]) - depth[mask]

SURFACE_MAX_M = 50
APOGEE_MAX_M = 20
BIN_M = 1
MIN_SAMPLES = 5

def binned_stats(distance, velocity, max_distance):
    centers = np.arange(BIN_M / 2, max_distance, BIN_M)
    rows = []
    for c in centers:
        v = velocity[(distance >= c - BIN_M / 2) & (distance < c + BIN_M / 2)]
        v = v[np.isfinite(v)]
        rows.append((np.mean(v), np.median(v), np.std(v),
                     np.percentile(v, 2.5), np.percentile(v, 97.5))
                    if len(v) >= MIN_SAMPLES else (np.nan,) * 5)
    return centers, np.asarray(rows)

fig, axes = plt.subplots(2, 2, figsize=(11, 9), sharex=True,
                         gridspec_kw={'height_ratios': [SURFACE_MAX_M, APOGEE_MAX_M]},
                         constrained_layout=True)
for col, (cast, cast_mask, color) in enumerate((
    ('Descent', glider.UPDN.values == -1, '#0072B2'),
    ('Ascent', glider.UPDN.values == 1, '#D55E00'),
)):
    for row, (distance, limit, ylabel) in enumerate((
        (depth, SURFACE_MAX_M, 'Depth below surface (m)'),
        (height_above_apogee, APOGEE_MAX_M, 'Height above apogee (m)'),
    )):
        y, stats = binned_stats(distance[cast_mask], glider.W_WATER.values[cast_mask], limit)
        mean, median, std, low, high = stats.T
        ax = axes[row, col]
        ax.plot(mean, y, color=color, label='Mean')
        ax.plot(median, y, '--', color=color, label='Median')
        ax.fill_betweenx(y, mean - std, mean + std, color=color, alpha=.2, label='±1 SD')
        ax.plot(low, y, ':', color='0.4', label='2.5–97.5%')
        ax.plot(high, y, ':', color='0.4')
        ax.set_ylim(limit, 0)
        ax.set_ylabel(ylabel)
        ax.grid(alpha=.25)
        ax.set_title(f'{cast}: {"surface" if row == 0 else "apogee"}')
        if row == 1:
            ax.set_xlabel('Calculated vertical water velocity (cm/s)')
        if row == 0 and col == 0:
            ax.legend(fontsize=8)
fig.suptitle('Vertical water velocity by cast and position')
plt.show()